# Compare data structures

Two flavours of pairwise comparison:

1. **`compare()`** — point ratios with deterministic CI envelope from the summary's stored CIs. Fast, no samples needed.
2. **`bootstrap_ratio_ci()`** — percentile-bootstrap CI on raw per-iteration samples. Tighter and statistically rigorous.

**Requires** ≥2 distinct `data_structure` values. The notebook auto-picks the first two it finds; override `BASELINE` and `TARGET` below to compare different pairs.

In [ ]:
import kermit_lab as kl

PATHS = '../../../bench-runs/*.json'
CRITERION_ROOT = '../../../target/criterion'

kl.apply_style()
df = kl.load(PATHS, criterion_root=CRITERION_ROOT)
samples = kl.load_samples(PATHS, criterion_root=CRITERION_ROOT)

ds_values = sorted(df.data_structure.dropna().unique().tolist())
assert len(ds_values) >= 2, f'need ≥2 data_structures; got {ds_values}'
BASELINE, TARGET = ds_values[0], ds_values[1]
print(f'comparing baseline={BASELINE!r} vs target={TARGET!r}')

## Pairwise speedup with envelope CIs

In [ ]:
iter_time = df[(df.metric == 'time') & (df.phase == 'iteration')]
result = kl.compare(iter_time, baseline=BASELINE, target=TARGET)
result[['query', 'tuples', 'speedup', 'speedup_lo', 'speedup_hi']]

## Bootstrap CI on raw samples (rigorous)

In [ ]:
labelled = samples.merge(
    df[['criterion_group', 'criterion_function', 'data_structure', 'phase']],
    on=['criterion_group', 'criterion_function'],
)
a = labelled.query(f"data_structure == @BASELINE and phase == 'iteration'").per_iter_ns.to_numpy()
b = labelled.query(f"data_structure == @TARGET and phase == 'iteration'").per_iter_ns.to_numpy()
lo, hi = kl.bootstrap_ratio_ci(a, b, rng=42)
print(f'{BASELINE} / {TARGET} speedup: {a.mean()/b.mean():.3f}  95% CI: [{lo:.3f}, {hi:.3f}]')